<a href="https://colab.research.google.com/github/student-shivamm/guarded-voice/blob/main/DEEPFAKE_DETECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

# This will ask you to choose a file.
# Pick the 'kaggle.json' file you just downloaded.
files.upload()

In [ ]:
# 1. Setup Kaggle configuration
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 2. Download the DEEP-VOICE dataset
print("Downloading dataset... (This may take a minute)")
!kaggle datasets download -d birdy654/deep-voice-deepfake-voice-recognition

# 3. Unzip the dataset
print("Unzipping dataset...")
!unzip -q deep-voice-deepfake-voice-recognition.zip -d deep_voice_data
print("Dataset ready!")

In [ ]:
import librosa
import numpy as np
import os
import pandas as pd
from tqdm import tqdm

def extract_features(file_path):
    # 1. Load the audio file
    # We use a fixed duration (e.g., 3 seconds) to ensure consistency,
    # though mean pooling handles variable lengths well.
    try:
        y, sr = librosa.load(file_path, duration=3, offset=0.5)
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

    # 2. Extract MFCCs (Static features)
    # n_mfcc=13 is standard for speech
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    # 3. Extract Delta (First derivative - speed of change)
    delta_mfcc = librosa.feature.delta(mfcc)

    # 4. Extract Delta-Delta (Second derivative - acceleration of change)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)

    # 5. Concatenate Features (Stack them vertically)
    # Shape becomes (39, Time_Steps) -> 13 MFCC + 13 Delta + 13 Delta2
    combined_features = np.vstack([mfcc, delta_mfcc, delta2_mfcc])

    # 6. Mean Pooling (Collapse time axis)
    # Shape becomes (39,) -> One single vector representing the whole audio
    feature_vector = np.mean(combined_features, axis=1)

    return feature_vector

In [ ]:
# Define paths (Based on the standard unzip structure of this dataset)
# Adjust these if your unzip path is slightly different
REAL_PATH = "/content/deep_voice_data/KAGGLE/AUDIO/REAL"
FAKE_PATH = "/content/deep_voice_data/KAGGLE/AUDIO/FAKE"

X = [] # Features
y = [] # Labels (0 = Real, 1 = Fake)

# Config: limit files for speed/memory in this demo
MAX_FILES_PER_CLASS = 5000

def select_audio_files(folder_path, limit):
    """Select sorted regular WAV/MP3 files before applying the candidate cap.

    The limit counts eligible candidates, including any later decoding failures.
    Sorting is deterministic for an unchanged directory; it is not random sampling.
    """
    if isinstance(limit, bool) or not isinstance(limit, int) or limit < 0:
        raise ValueError('limit must be a non-negative integer')
    eligible = [
        name for name in os.listdir(folder_path)
        if os.path.splitext(name)[1].lower() in {'.wav', '.mp3'}
        and os.path.isfile(os.path.join(folder_path, name))
    ]
    return sorted(eligible)[:limit]


def process_folder(folder_path, label):
    print(f"Processing {folder_path}...")
    files = select_audio_files(folder_path, MAX_FILES_PER_CLASS)
    print(f"Selected {len(files)} audio candidates in filename order.")
    for file in tqdm(files):
        path = os.path.join(folder_path, file)
        features = extract_features(path)
        if features is not None:
            X.append(features)
            y.append(label)

# Process both classes
process_folder(REAL_PATH, label=0)
process_folder(FAKE_PATH, label=1)

# Convert to Numpy arrays
X = np.array(X)
y = np.array(y)

print(f"\nFeature Matrix Shape: {X.shape}") # Should be (Total_Samples, 39)
print(f"Labels Shape: {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Scale Features (Crucial for SVM!)
# SVM calculates distances, so features must be on the same scale.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Train SVM
print("Training SVM Model...")
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale') # RBF kernel is standard for audio
svm_model.fit(X_train_scaled, y_train)
print("Training Complete.")

In [ ]:
# 1. Predict
y_pred = svm_model.predict(X_test_scaled)

# 2. Metrics
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))

# 3. Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
import numpy as np

# Assuming you have X and y from the previous step
if 'y' in locals() and len(y) > 0:
    y_arr = np.array(y)
    real_count = np.sum(y_arr == 0)
    fake_count = np.sum(y_arr == 1)

    print("📊 DATASET BALANCE CHECK")
    print("="*30)
    print(f"🟢 Real Samples: {real_count}")
    print(f"🔴 Fake Samples: {fake_count}")
    print(f"🏆 Total Samples: {len(y_arr)}")
    print("="*30)

    if real_count < 50:
        print("⚠️ CRITICAL ERROR: You have almost NO Real data.")
        print("   The model ignores 'Real' because it barely exists.")
        print("   👉 Solution: You MUST upload more Real audio or fix the Audio Chopper.")
    elif fake_count > (real_count * 2):
        print("⚠️ IMBALANCE DETECTED: You have way more Fakes than Real.")
        print("   👉 Solution: The model is biased. Upload more Real files to balance it.")
    else:
        print("✅ Balance looks acceptable. The issue might be random chance in the test split.")
else:
    print("❌ Error: No data loaded in memory (y). Run the training block first.")

In [ ]:
# @title ✂️ Chunk REAL Audio into 30-Second Segments
import os
import librosa
import soundfile as sf
from tqdm import tqdm

# --- CONFIGURATION ---
TARGET_FOLDER = "/content/deep_voice_data/KAGGLE/AUDIO/REAL"
CHUNK_DURATION = 30.0  # Seconds
MIN_DURATION = 3.0     # Ignore leftovers smaller than 3 seconds

def chunk_real_audio():
    print(f"🔄 Processing folder: {TARGET_FOLDER}")

    if not os.path.exists(TARGET_FOLDER):
        print("❌ Error: REAL folder not found. Please check your path.")
        return

    # Get list of files
    # We filter for .wav/.mp3 and exclude files that are already chunks to avoid re-chopping
    files = [f for f in os.listdir(TARGET_FOLDER)
             if (f.endswith('.wav') or f.endswith('.mp3')) and "_30s_chunk_" not in f]

    if len(files) == 0:
        print("⚠️ No new files found to chop.")
        return

    print(f"   Found {len(files)} files to process...")
    new_files_count = 0

    for filename in tqdm(files):
        file_path = os.path.join(TARGET_FOLDER, filename)

        try:
            # Load audio (sr=None preserves original quality)
            y, sr = librosa.load(file_path, sr=None)
            total_duration = librosa.get_duration(y=y, sr=sr)

            # If file is shorter than target, just keep it (maybe rename for consistency)
            if total_duration < CHUNK_DURATION:
                continue

            # CALCULATE CHUNKS
            chunk_length_samples = int(CHUNK_DURATION * sr)
            total_samples = len(y)

            # SPLIT & SAVE
            for i in range(0, total_samples, chunk_length_samples):
                chunk = y[i : i + chunk_length_samples]

                # Check if the chunk is big enough to be useful
                if len(chunk) < (MIN_DURATION * sr):
                    continue

                # Create new filename
                # Example: "interview_10s_chunk_0.wav"
                chunk_name = f"{os.path.splitext(filename)[0]}_10s_chunk_{i//chunk_length_samples}.wav"
                out_path = os.path.join(TARGET_FOLDER, chunk_name)

                sf.write(out_path, chunk, sr)
                new_files_count += 1

            # OPTIONAL: Remove the original long file to save space/confusion
            os.remove(file_path)

        except Exception as e:
            print(f"   ⚠️ Error processing {filename}: {e}")

    print(f"\n✅ SUCCESS! Created {new_files_count} new 10-second clips.")
    print("👉 Note: Original long files have been removed.")

# Run it
chunk_real_audio()

In [ ]:
from google.colab import files
import numpy as np
from sklearn.svm import SVC

# --- PART 1: UPDATE THE MODEL FOR PROBABILITIES ---
# We need to re-train the SVM with 'probability=True' to get confidence scores.
# This uses the data (X, y) you already loaded in the previous step.

if 'X' in locals() and len(X) > 0:
    print("Updating model to support confidence scores...")

    # Re-scale data (Safety step to ensure consistency)
    X_arr = np.array(X)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_arr)

    # TRAIN with probability=True
    clf = SVC(kernel='rbf', probability=True) # <--- THIS IS THE KEY CHANGE
    clf.fit(X_scaled, y)
    print("Model updated! Now supports confidence percentages.\n")

    # --- PART 2: UPLOAD AND PREDICT WITH CONFIDENCE ---
    print("--- UPLOAD YOUR FILE TO TEST ---")
    uploaded = files.upload()

    for filename in uploaded.keys():
        print(f"\nAnalyzing: {filename}...")

        # Extract features
        new_features = extract_features(filename)

        if new_features is not None:
            # Prepare data
            new_features = new_features.reshape(1, -1)
            new_features_scaled = scaler.transform(new_features)

            # GET PROBABILITIES (The Confidence Scores)
            # probs[0][0] = Probability of Real (Class 0)
            # probs[0][1] = Probability of Fake (Class 1)
            probs = clf.predict_proba(new_features_scaled)

            real_score = probs[0][0] * 100
            fake_score = probs[0][1] * 100

            # --- DISPLAY RESULTS ---
            print("\n" + "="*40)
            if fake_score > real_score:
                print(f"🔴 RESULT: FAKE / AI VOICE")
                print(f"   Confidence: {fake_score:.2f}%")
            else:
                print(f"🟢 RESULT: REAL HUMAN VOICE")
                print(f"   Confidence: {real_score:.2f}%")
            print("="*40)

            # Detailed breakdown
            print(f"\nDetailed Breakdown:")
            print(f"Real Voice Probability: {real_score:.2f}%")
            print(f"Fake Voice Probability: {fake_score:.2f}%")

        else:
            print("Error: Could not extract features from this file.")

else:
    print("CRITICAL ERROR: Training data 'X' not found.")
    print("Please run the previous training block (Step 3) first to load the data.")

In [ ]:
# --- ADD YOUR OWN VOICE TO TRAINING ---
from google.colab import files
import os
import shutil

# 1. Upload your REAL voice samples
print("Upload 5-10 recordings of your OWN voice (Real) to teach the AI:")
uploaded = files.upload()

# 2. Move them to the REAL folder
real_folder = "/content/deep_voice_data/KAGGLE/AUDIO/REAL"
print(f"\nAdding your voices to: {real_folder}")

count = 0
for filename in uploaded.keys():
    # Rename to avoid conflicts
    destination = os.path.join(real_folder, f"my_voice_{count}.wav")
    shutil.move(filename, destination)
    count += 1
print(f"Added {count} new files to the REAL dataset.")

# 3. RETRAIN THE MODEL (Step 3 code again)
print("\nRetraining model with your voice included...")
# (Using the logic from before, but now it includes your files)

X = []
y = []
MAX_FILES = 1200 # Increased slightly to include your files

# Load Real Data
for f in os.listdir(REAL_PATH)[:MAX_FILES]:
    path = os.path.join(REAL_PATH, f)
    feats = extract_features(path) # Uses the NEW extract_features from Solution 1
    if feats is not None:
        X.append(feats)
        y.append(0) # 0 = Real

# Load Fake Data
for f in os.listdir(FAKE_PATH)[:MAX_FILES]:
    path = os.path.join(FAKE_PATH, f)
    feats = extract_features(path)
    if feats is not None:
        X.append(feats)
        y.append(1) # 1 = Fake

# Train
X_arr = np.array(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_arr)
clf = SVC(kernel='rbf', probability=True)
clf.fit(X_scaled, y)

print("SUCCESS! Model is now smarter. Try testing again.")

In [ ]:
import numpy as np
import os
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# Paths
REAL_PATH = "/content/deep_voice_data/KAGGLE/AUDIO/REAL"
FAKE_PATH = "/content/deep_voice_data/KAGGLE/AUDIO/FAKE"

print("🔄 Retraining model with your new data included...")

# 1. Load Data (Increased limit to catch your new files)
X = []
y = []
MAX_FILES = 1500

def load_data(folder, label, label_name):
    print(f"   Loading {label_name} files...")
    files = os.listdir(folder)
    # Sort to ensure we get a mix, but prioritize your new files if possible
    # (Your new files usually are at the end, so we read everything we can)
    count = 0
    for f in tqdm(files):
        if count >= MAX_FILES: break

        path = os.path.join(folder, f)
        # Use the robust extraction function
        feats = extract_features(path)
        if feats is not None:
            X.append(feats)
            y.append(label)
            count += 1

# Load (0=Real, 1=Fake)
load_data(REAL_PATH, 0, "REAL")
load_data(FAKE_PATH, 1, "FAKE")

# 2. Train SVM
print("\n🧠 Fitting the AI Model...")
if len(X) > 0:
    X_arr = np.array(X)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_arr)

    # Probability=True is needed for the % score
    clf = SVC(kernel='rbf', probability=True)
    clf.fit(X_scaled, y)
    print("✅ SUCCESS: Model Retrained!")
else:
    print("❌ ERROR: No data found. Check your folders.")

In [ ]:
from google.colab import files
import shutil
import os

fake_folder = "/content/deep_voice_data/KAGGLE/AUDIO/FAKE"

print("--- UPLOAD FAKE AUDIO FILES (.wav or .mp3) ---")
uploaded = files.upload()

for filename in uploaded.keys():
    # Move file to the FAKE folder
    destination = os.path.join(fake_folder, filename)
    shutil.move(filename, destination)
    print(f"✅ Added to FAKE dataset: {filename}")

In [ ]:
# @title ✂️ Universal 5-Second Chopper (Real & Fake)
import os
import librosa
import soundfile as sf
from tqdm import tqdm
import math

# --- CONFIGURATION ---
REAL_PATH = "/content/deep_voice_data/KAGGLE/AUDIO/REAL"
FAKE_PATH = "/content/deep_voice_data/KAGGLE/AUDIO/FAKE"
CHUNK_DURATION = 5.0   # Target length in seconds
MIN_DURATION = 2.0     # Discard leftovers smaller than this

def chunk_all_files(folder_path, label):
    print(f"\n🔄 Processing {label} folder: {folder_path}")

    if not os.path.exists(folder_path):
        print(f"❌ Error: {label} folder not found.")
        return

    # Gather all audio files
    # We exclude files that are already labeled '_5s_chunk_' to prevent double-processing
    all_files = [f for f in os.listdir(folder_path)
                 if (f.endswith('.wav') or f.endswith('.mp3')) and "_5s_chunk_" not in f]

    if len(all_files) == 0:
        print("   ⚠️ No new files found to chop.")
        return

    print(f"   Found {len(all_files)} files to chop into 5s segments...")
    new_chunks_count = 0

    for filename in tqdm(all_files):
        file_path = os.path.join(folder_path, filename)

        try:
            # Load the full audio file (sr=None keeps original quality)
            y, sr = librosa.load(file_path, sr=None)
            total_duration = librosa.get_duration(y=y, sr=sr)

            # Calculate how many chunks we can make
            chunk_length_samples = int(CHUNK_DURATION * sr)
            total_samples = len(y)

            # SLICING LOOP
            for i in range(0, total_samples, chunk_length_samples):
                chunk = y[i : i + chunk_length_samples]

                # Check if the chunk is valid (not too short)
                if len(chunk) < (MIN_DURATION * sr):
                    continue

                # Save the new chunk
                # Name format: originalName_5s_chunk_0.wav
                chunk_name = f"{os.path.splitext(filename)[0]}_5s_chunk_{i//chunk_length_samples}.wav"
                out_path = os.path.join(folder_path, chunk_name)

                sf.write(out_path, chunk, sr)
                new_chunks_count += 1

            # DELETE ORIGINAL (To save space and avoid duplicates)
            os.remove(file_path)

        except Exception as e:
            print(f"   ⚠️ Failed to process {filename}: {e}")

    print(f"✅ Finished {label}! Created {new_chunks_count} new 5-second clips.")

# --- EXECUTE ---
chunk_all_files(REAL_PATH, "REAL")
chunk_all_files(FAKE_PATH, "FAKE")

In [ ]:
import numpy as np
import os
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# Define Paths
REAL_PATH = "/content/deep_voice_data/KAGGLE/AUDIO/REAL"
FAKE_PATH = "/content/deep_voice_data/KAGGLE/AUDIO/FAKE"

print("🔄 Retraining model with NEW FAKE Data...")

X = []
y = []
# INCREASE THIS NUMBER to make sure it reads your new files!
MAX_FILES = 1500

# 1. Load Real Data (Label 0)
print("   Reading Real files...")
for f in tqdm(os.listdir(REAL_PATH)[:MAX_FILES]):
    path = os.path.join(REAL_PATH, f)
    feats = extract_features(path)
    if feats is not None:
        X.append(feats)
        y.append(0)

# 2. Load Fake Data (Label 1) - Now includes your generated files
print("   Reading Fake files...")
for f in tqdm(os.listdir(FAKE_PATH)[:MAX_FILES]):
    path = os.path.join(FAKE_PATH, f)
    feats = extract_features(path)
    if feats is not None:
        X.append(feats)
        y.append(1)

# 3. Train
if len(X) > 0:
    X_arr = np.array(X)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_arr)

    clf = SVC(kernel='rbf', probability=True)
    clf.fit(X_scaled, y)
    print("\n🎉 Model Retrained! It has now studied your new Fake voices.")
else:
    print("Error: No data found.")

In [ ]:
from google.colab import files

print("--- UPLOAD A NEW TEST FILE ---")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nAnalyzing: {filename}...")
    new_features = extract_features(filename)

    if new_features is not None:
        # Reshape & Scale
        new_features = new_features.reshape(1, -1)
        new_features_scaled = scaler.transform(new_features)

        # Predict
        probs = clf.predict_proba(new_features_scaled)
        real_score = probs[0][0] * 100
        fake_score = probs[0][1] * 100

        print("\n" + "="*40)
        if fake_score > real_score:
            print(f"🔴 RESULT: FAKE / AI VOICE")
            print(f"   Confidence: {fake_score:.2f}%")
        else:
            print(f"🟢 RESULT: REAL HUMAN VOICE")
            print(f"   Confidence: {real_score:.2f}%")
        print("="*40)

In [ ]:
# @title ⚖️ ULTIMATE COURTROOM SCANNER (No Silence Removal + 5s Logic)
import librosa
import numpy as np
from google.colab import files
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
CHUNK_DURATION = 5.0   # 5 Seconds (Matches your new training chunks)
OVERLAP = 2.0          # Overlap to ensure no word is cut in half

if 'clf' not in locals():
    print("⚠️ ERROR: Please load the model ('clf') first!")
else:
    print("--- UPLOAD AUDIO (Full Raw Analysis) ---")
    uploaded = files.upload()

    for filename in uploaded.keys():
        print(f"\n" + "="*70)
        print(f"⚡ Analyzing Evidence: {filename}")

        try:
            # 1. Load Full Audio (Raw, No Silence Removal)
            y, sr = librosa.load(filename, sr=22050)
            duration = librosa.get_duration(y=y, sr=sr)
            print(f"   Audio Length: {duration:.2f} seconds")
            print("-" * 70)

        except Exception as e:
            print(f"Error loading: {e}")
            continue

        # 2. SLIDING WINDOW SCAN
        step = int((CHUNK_DURATION - OVERLAP) * sr)
        chunk_len = int(CHUNK_DURATION * sr)

        window_scores = []
        suspicious_segments = []

        print(f"   {'TIME WINDOW':<15} | {'FAKE SCORE':<12} | {'STATUS'}")
        print("-" * 70)

        for start in range(0, len(y), step):
            end = start + chunk_len

            # Handle padding
            if end > len(y):
                if len(y) - start < 1.0 * sr: break
                seg = np.pad(y[start:], (0, chunk_len - (len(y)-start)), mode='wrap')
            else:
                seg = y[start:end]

            # Extract Features (Matches Training Exactly)
            mfcc = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=13)
            combined = np.vstack([mfcc, librosa.feature.delta(mfcc), librosa.feature.delta(mfcc, order=2)])
            feat = np.mean(combined, axis=1).reshape(1, -1)

            # Predict
            probs = clf.predict_proba(scaler.transform(feat))
            fake_prob = probs[0][1] * 100
            window_scores.append(fake_prob)

            # Calculate Time
            start_sec = start / sr
            time_str = f"{int(start_sec//60)}:{int(start_sec%60):02d}"

            if fake_prob > 80:
                status = "🔴 HIGH RISK"
            elif fake_prob > 50:
                status = "🟠 SUSPICIOUS"
            else:
                status = "🟢 OK"

            # Print only non-perfect segments to keep report clean
            if fake_prob > 30:
                print(f"   {time_str:<15} | {fake_prob:.2f}%       | {status}")

        # 3. FINAL LOGIC TREE
        if not window_scores:
            print("Audio too short.")
            continue

        max_score = max(window_scores)
        avg_score = sum(window_scores) / len(window_scores)

        print("-" * 70)
        print(f"📊 REPORT SUMMARY:")
        print(f"   Max Fake Spike: {max_score:.2f}%")
        print(f"   Average Score:  {avg_score:.2f}%")

        # --- DECISION LOGIC ---
        if max_score > 90.0:
            verdict = "🔴 FAKE"
            reason = "Found clear synthetic artifacts (>90%)."

        # Hidden Spike Rule (The TTS Catcher)
        elif max_score > 75.0:
            verdict = "🟠 SUSPICIOUS / POTENTIAL FAKE"
            reason = "High spike detected (likely TTS or splice)."

        # General Consistency Rule
        elif avg_score > 55.0:
             verdict = "🔴 FAKE"
             reason = "Overall audio quality suggests synthetic generation."

        # Safe Rule
        else:
            verdict = "🟢 REAL"
            reason = "No significant fake patterns found."

        print(f"\n🏆 FINAL VERDICT: {verdict}")
        print(f"📝 REASON: {reason}")
        print("="*70)

In [ ]:
import joblib
from google.colab import files

print("📦 Packaging your AI model for download...")

# 1. Check if model exists
if 'clf' in locals() and 'scaler' in locals():
    # 2. Save to Colab's temporary disk
    model_filename = "Rdeepfake_svm_model.pkl"
    scaler_filename = "Rdeepfake_scaler.pkl"

    joblib.dump(clf, model_filename)
    joblib.dump(scaler, scaler_filename)

    print(f"✅ Saved internal files: {model_filename}, {scaler_filename}")

    # 3. Download to your computer
    print("⬇️ Triggering download... (Check your browser downloads)")
    files.download(model_filename)
    files.download(scaler_filename)

    print("\n🎉 DONE! Keep these files safe.")
else:
    print("❌ ERROR: Model ('clf') or Scaler ('scaler') not found. Did you run the training step?") #we will insert here samples report with the test samples

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Prepare Data (Assumes X and y are already loaded from your previous steps)
# If X is not ready, run the 'load_data' block again first.

if 'X' in locals() and len(X) > 0:
    X_arr = np.array(X)
    y_arr = np.array(y)

    # 2. Split Data (80% for Study, 20% for Exam)
    X_train, X_test, y_train, y_test = train_test_split(X_arr, y_arr, test_size=0.2, random_state=42)

    # 3. Train on Study Data ONLY
    print("Training on 80% of data...")
    # Using probability=True as you requested before
    clf_check = SVC(kernel='rbf', probability=True)

    # Scale based only on training data to be strict
    scaler_check = StandardScaler()
    X_train_scaled = scaler_check.fit_transform(X_train)
    X_test_scaled = scaler_check.transform(X_test)

    clf_check.fit(X_train_scaled, y_train)

    # 4. Check Scores
    train_pred = clf_check.predict(X_train_scaled)
    test_pred = clf_check.predict(X_test_scaled)

    train_acc = accuracy_score(y_train, train_pred) * 100
    test_acc = accuracy_score(y_test, test_pred) * 100

    print("\n" + "="*30)
    print(f"📊 DIAGNOSTIC REPORT")
    print("="*30)
    print(f"Total Files: {len(X)}")
    print(f"Training Accuracy: {train_acc:.2f}% (How well it memorized)")
    print(f"Testing Accuracy:  {test_acc:.2f}% (How well it actually works)")

    gap = train_acc - test_acc
    print(f"Gap: {gap:.2f}%")

    print("\n🔍 ANALYSIS:")
    if gap > 15:
        print("⚠️ DANGER: OVERFITTING DETECTED.")
        print("The model is memorizing data but failing on new files.")
        print("SOLUTION: Add more diverse data (YouTube, Friends)!")
    elif gap < -5:
        print("❓ WEIRD: Underfitting or data mismatch.")
    else:
        print("✅ HEALTHY: The model is learning generalized patterns.")
        print("You can keep adding data!")
else:
    print("Error: No data loaded in 'X'. Run the loading/training block first.")

In [ ]:
import joblib
from google.colab import files

# 1. Upload the 2 .pkl files
print("--- UPLOAD YOUR SAVED MODEL & SCALER ---")
uploaded = files.upload()

# 2. Load them back into memory
try:
    # Load the Model
    clf = joblib.load("Rdeepfake_svm_model (1).pkl")
    # Load the Scaler
    scaler = joblib.load("Rdeepfake_scaler (1).pkl")

    print("\n✅ Success! Model loaded.")
    print("🚀 You can skip training and go straight to the 'Sliding Window Scanner' code!")

except Exception as e:
    print(f"\n❌ Error: {e}")
    print("Make sure you uploaded both 'deepfake_svm_model.pkl' and 'deepfake_scaler.pkl'")

In [ ]:
import os

# Define Paths
real_path = "/content/deep_voice_data/KAGGLE/AUDIO/REAL"
fake_path = "/content/deep_voice_data/KAGGLE/AUDIO/FAKE"

def count_files(folder_path, label):
    if os.path.exists(folder_path):
        files = os.listdir(folder_path)
        count = len(files)
        print(f"📂 {label} Folder: {count} files")
        return count
    else:
        print(f"❌ {label} Folder not found!")
        return 0

print("--- 📊 DATASET REPORT ---")
real_count = count_files(real_path, "REAL")
fake_count = count_files(fake_path, "FAKE")

print("-" * 30)
total = real_count + fake_count
print(f"🏆 TOTAL FILES: {total}")

if total > 0:
    print("-" * 30)
    # Check Balance
    diff = abs(real_count - fake_count)
    if diff > (total * 0.1): # If difference is > 10%
        print("⚠️ IMBALANCED DATA DETECTED!")